# ⚙️ MLOps Pipeline Notebook
## Experiment Tracking · Hyperparameter Search · Model Lifecycle

```mermaid
flowchart TD
    A[Data] --> B[HPO Search
Optuna]
    B --> C[Best Config]
    C --> D[Full Training
MLflow Tracking]
    D --> E{Val AUC
> threshold?}
    E -- Yes --> F[Register Champion]
    E -- No  --> G[Retire Candidate]
    F --> H[Deploy API]
```


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np
print("Ready ✓")


## 1. Data

In [ ]:
from src.data.pipeline import ChurnDataPipeline, PipelineConfig
pipeline = ChurnDataPipeline(PipelineConfig(save_artifacts=True, artifacts_dir='../artifacts/pipeline'))
split = pipeline.run('../data/raw/ChurnPrediction.csv')
print(f"Train={split.train_size} Val={split.val_size} Test={split.test_size}")


## 2. Hyperparameter Search (Optuna)

In [ ]:
# NOTE: Set n_trials=3 for quick demo; increase to 50+ for real search
try:
    import optuna
    from src.training.hyperparameter_search import HyperparameterSearch, SearchConfig

    search = HyperparameterSearch(SearchConfig(
        n_trials=3,
        max_epochs=20,
        metric='val_auc',
        output_dir='../artifacts/hpo',
    ))
    best_params = search.run(split.X_train, split.y_train, split.X_val, split.y_val)
    print("Best params:", best_params)
except ImportError:
    print("Optuna not installed — using default config")
    best_params = None


## 3. Training with MLflow

In [ ]:
from src.models.ann import ChurnANN, baseline_config, regularised_config
from src.training.trainer import ChurnModelTrainer, TrainingConfig

cfg = regularised_config(split.X_train.shape[1])
model = ChurnANN(cfg).build()

trainer = ChurnModelTrainer(TrainingConfig(
    epochs=50,
    batch_size=32,
    early_stopping_patience=10,
    enable_mlflow=False,   # set True if mlflow server running
    checkpoint_dir='../artifacts/checkpoints',
    verbose=0,
))
result = trainer.train(model, split.X_train, split.y_train, split.X_val, split.y_val,
                       model_config=cfg.to_dict())
print(f"Val AUC: {result.best_val_auc:.4f} | Val Loss: {result.best_val_loss:.4f}")


## 4. Evaluation & Registry Promotion

In [ ]:
from src.evaluation.evaluator import ModelEvaluator
from src.utils.model_registry import ModelRegistry

evaluator = ModelEvaluator(output_dir='../artifacts/evaluation')
eval_result = evaluator.evaluate(model, split.X_test, split.y_test, split='test')
evaluator.print_report(eval_result)

registry = ModelRegistry(root='../artifacts/model_registry')
entry = registry.register(
    name='churn-ann',
    model_path=result.model_path,
    metrics={
        'val_auc': result.best_val_auc,
        'test_roc_auc': eval_result.roc_auc,
        'test_f1': eval_result.f1,
    },
    tags={'preset': 'regularised'},
)

AUC_THRESHOLD = 0.80
if eval_result.roc_auc >= AUC_THRESHOLD:
    registry.promote(entry.model_id, stage='champion')
    print(f"✓ Model promoted to champion (AUC={eval_result.roc_auc:.4f})")
else:
    print(f"✗ Model not promoted (AUC={eval_result.roc_auc:.4f} < {AUC_THRESHOLD})")

registry.print_registry()


## 5. Monitoring Setup

In [ ]:
from src.monitoring.monitoring_service import MonitoringService
import numpy as np

monitor = MonitoringService(output_dir='../artifacts/monitoring')
monitor.set_reference_distribution(split.X_train, feature_names=[
    'CreditScore','Geography','Gender','Age','Tenure',
    'Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary'
])

# Simulate 200 predictions
y_proba = model.predict(split.X_test, verbose=0).ravel()
import time
for i, (prob, feat) in enumerate(zip(y_proba[:200], split.X_test[:200])):
    monitor.record_prediction(
        probability=float(prob),
        latency_ms=np.random.exponential(4.0),
        features=feat.reshape(1,-1),
    )

report = monitor.get_report()
print(f"Requests       : {report['requests']['total']}")
print(f"Churn rate     : {report['predictions']['churn_rate']:.1%}")
print(f"Mean latency   : {report['latency_ms']['mean']:.2f}ms")
print(f"p99 latency    : {report['latency_ms']['p99']:.2f}ms")
monitor.save_report()
print("Monitoring report saved.")
